# Principal Component Analysis (PCA)

Welcome to the eleventh notebook in our repository. This notebook covers everything from intuition to mathematical foundations and practical Scikit-learn workflows.

==================================================

# 1. What is Dimensionality Reduction?

==================================================

**What is dimensionality?**
Dimensionality is simply the number of features (variables) in your data. If you have a dataset with 5 columns, it's 5-dimensional.

**What does a feature represent?**
A feature represents an independent, measurable property of the data. For instance, in a dataset of cars, the features could be "Horsepower", "Weight", and "Price".

**Why too many features can be a problem**
While more features can mean more information, they can also cause the **Curse of Dimensionality**:
- **Harder visualization:** Humans can only visualize up to 3 dimensions. Finding patterns in 50 dimensions is impossible visually.
- **More computation:** Models take much longer to train on massive datasets and consume more memory.
- **Possible noise/redundancy:** Features are often correlated (e.g., house size in sq ft and house size in sq meters). Having both adds no new information but confuses algorithms.

**What is dimensionality reduction?**
Dimensionality reduction transforms a dataset with many features into one with fewer features, while attempting to retain as much variance (important information) as possible.

**Why dimensionality reduction is useful**
Many Features $\rightarrow$ Harder visualization $\rightarrow$ More computation $\rightarrow$ Possible noise/redundancy $\rightarrow$ Reduce dimensions $\rightarrow$ Keep important information

==================================================

# 2. What is PCA?

==================================================

**What is Principal Component Analysis?**
PCA is a mathematical technique used for feature extraction and dimensionality reduction.

**Why PCA is used**
It simplifies complex, high-dimensional datasets while keeping the underlying structure, relationships, and variance intact.

**Main intuition**
Imagine taking a 2D photograph of a 3D object. Depending on the angle you choose, you can capture more or less detail. PCA finds the mathematical "angles" (directions) that capture the maximum amount of variation in the data.

**PCA as an unsupervised technique**
PCA does not use target labels to create components! It looks entirely at how the features ($X$) relate to one another, completely ignoring the target variable ($y$).

**Simple example:**
10 features $\rightarrow$ PCA $\rightarrow$ 2 principal components.

**Where PCA is useful:**
- **Visualization:** Reducing hundreds of features to 2 or 3 components so they can be plotted.
- **Noise reduction:** Dropping features that don't explain much variance removes noise.
- **Feature compression:** Condensing data to save space.
- **Faster ML models:** Reducing input complexity.
- **Removing redundancy:** Combining tightly correlated features into single principal components.

==================================================

# 3. Intuition Behind PCA

==================================================

Imagine a simple 2D dataset with two features.
- **Original feature axes:** Standard X and Y representing Feature 1 and Feature 2.
- **Direction of maximum variation:** The diagonal direction where the data is most spread out.
- **First principal component:** The primary axis drawn exactly through this direction of maximum variation.
- **Second principal component:** An axis orthogonal (perpendicular) to the first component.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")
np.random.seed(42)

# Create simple 2D dataset with strong correlation
X_2d = np.dot(np.random.rand(2, 2), np.random.randn(2, 200)).T

pca_vis = PCA(n_components=2)
pca_vis.fit(X_2d)

plt.figure(figsize=(8, 6))
plt.scatter(X_2d[:, 0], X_2d[:, 1], alpha=0.6, label='Data points')

# Plot the PCA arrows
for length, vector in zip(pca_vis.explained_variance_, pca_vis.components_):
    v = vector * 3 * np.sqrt(length) # Scale vector for visualization
    plt.annotate('', pca_vis.mean_ + v, pca_vis.mean_,
                 arrowprops=dict(arrowstyle='->', linewidth=2, shrinkA=0, shrinkB=0, color='red'))

plt.title('Data Points with Principal Component Directions')
plt.xlabel('Feature 1 (Original axes)')
plt.ylabel('Feature 2 (Original axes)')
plt.axis('equal')
plt.legend()
plt.show()

==================================================

# 4. Variance and PCA

==================================================

> **Key Idea:** PCA tries to capture high variance.

**What variance means**
Variance relates to how "spread out" numerical data is.

**Why high variance may contain useful information**
If a feature does not vary across samples (e.g., everyone has 2 eyes), it provides no distinguishing power for a machine learning model! Directions with maximum spread generally hide the most distinguishing signals.

**Why PCA looks for directions with maximum variance**
By isolating directions of maximum variation and ignoring directions where the data is flat, PCA compresses data down to its most mathematically expressive components.

**Small numerical example:**
Think of "Height" vs "Number of heads". Height varies widely (high variance) allowing us to distinguish people. Number of heads is always 1 (zero variance). PCA ignores zero variance.

==================================================

# 5. Principal Components

==================================================

- **PC1:** The very first Principal Component. It is positioned along the single axis that captures the absolute maximum possible variance.
- **PC2:** The second component. It MUST be orthogonal (perpendicular) to PC1, and captures the maximum amount of the *remaining* variance.
- **PC3:** Orthogonal to both PC1 and PC2, capturing the next largest slice of variance, and so on.

**Components are combinations of original features**
PCA doesn't just pick "Feature A" and "Feature B". It constructs new variables. PC1 might literally be $0.80 \times FeatureA + 0.20 \times FeatureB$.

> **Important:** Components are **orthogonal/uncorrelated**. The new features (PC1, PC2) will have zero correlation with each other!

==================================================

# 6. PCA Mathematics — Intuition

==================================================

The high-level math works as follows:

1. **Mean centering:** Shift the data so the center is exactly at the origin (0, 0).
2. **Covariance:** Build a Covariance Matrix assessing how every feature relates to every other feature.
3. **Eigenvectors:** Decompose this matrix to find the **Eigenvectors**. These essentially represent the **directions** of the new feature space!
4. **Eigenvalues:** Each eigenvector has an **Eigenvalue**. The eigenvalue represents the **amount of variance captured** by its corresponding eigenvector.

**Relationship:**
- **Eigenvectors** $\rightarrow$ Directions
- **Eigenvalues** $\rightarrow$ Amount of variance captured

==================================================

# 7. Standardization Before PCA

==================================================

> **Common Mistake:** Applying PCA without scaling features first!

**Why scaling is usually performed before PCA**
PCA computes variance. If one feature is measured in millions (e.g., Salary) and another in single digits (e.g., Years of Experience), PCA will just automatically assume "Salary" is the most important component purely because of its massive numerical scale.

Standardizing transforms features to have a mean of 0 and a variance of 1, allowing PCA to legitimately compare structural relevance.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Demonstrate scaling
unscaled_data = np.array([
    [100, 1],
    [500, 2],
    [300, 1],
    [800, 3]
])

print("Unscaled Feature 1 Variance:", np.var(unscaled_data[:, 0]))
print("Unscaled Feature 2 Variance:", np.var(unscaled_data[:, 1]))
print("-" * 40)

scaler = StandardScaler()
scaled_data = scaler.fit_transform(unscaled_data)

print("Scaled Feature 1 Variance:", np.var(scaled_data[:, 0]))
print("Scaled Feature 2 Variance:", np.var(scaled_data[:, 1]))

Now PCA can fairly evaluate both features.

==================================================

# 8. PCA from Scratch — Simple Version

==================================================

Let's do the simplest version of PCA from scratch using pure NumPy before letting Scikit-learn do the heavy lifting.

In [ ]:
# Create small 2D dataset
X_scratch = np.random.randn(100, 2)
X_scratch[:, 1] = X_scratch[:, 0] * 2 + np.random.randn(100) * 0.5 # High correlation

# 1. Center data
X_centered = X_scratch - np.mean(X_scratch, axis=0)

# 2. Calculate covariance matrix
cov_matrix = np.cov(X_centered, rowvar=False)

# 3. Calculate eigenvalues/eigenvectors
eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)

# 4. Sort components
sorted_idx = np.argsort(eigenvalues)[::-1]
eigenvalues_sorted = eigenvalues[sorted_idx]
eigenvectors_sorted = eigenvectors[:, sorted_idx]

# 5. Select principal components (Top 1)
top_components = eigenvectors_sorted[:, :1]

# 6. Project data
X_projected = np.dot(X_centered, top_components)

print("Original shape:", X_scratch.shape)
print("Projected shape (1D):", X_projected.shape)
print("Eigenvalues (variance):", eigenvalues_sorted)

==================================================

# 9. PCA with Scikit-learn

==================================================

Now let's use the industry standard approach.

In [ ]:
from sklearn.decomposition import PCA

In [ ]:
# Standardize first!
X_scaled = StandardScaler().fit_transform(X_scratch)

# Apply PCA
pca = PCA(n_components=1)
X_pca = pca.fit_transform(X_scaled)

**Explanation:**
- `n_components`: How many components to keep.
- `fit()`: Learns the eigenvectors and eigenvalues.
- `transform()`: Projects the original data onto the principal components.
- `fit_transform()`: Does both `fit` and `transform` in one convenient step.

==================================================

# 10. Explained Variance

==================================================

- **Explained variance:** The actual raw eigenvalue amounts.
- **Explained variance ratio:** The % of the dataset's total variance covered by a component.
- **Cumulative explained variance:** The running total ratio of variance explained.

In [ ]:
pca_full = PCA(n_components=2)
pca_full.fit(X_scaled)

print("Explained variance:", pca_full.explained_variance_)
print("Explained variance ratio:", pca_full.explained_variance_ratio_)

In [ ]:
variance_table = pd.DataFrame({
    'Component': ['PC1', 'PC2'],
    'Variance': pca_full.explained_variance_,
    'Explained Variance Ratio': pca_full.explained_variance_ratio_
})
display(variance_table)

==================================================

# 11. Choosing Number of Components

==================================================

We usually choose enough components to preserve most of the variance (e.g., 90%, 95%, or 99%).
> **Remember:** There is no universal percentage threshold; it heavily depends on your specific problem!

In [ ]:
# Create dummy 10-dimensional data
X_large = np.random.randn(300, 10)
# Add arbitrary correlations
X_large[:, 2] = X_large[:, 1] * 3
X_large[:, 5] = X_large[:, 0] * 2

pca_curve = PCA()
pca_curve.fit(StandardScaler().fit_transform(X_large))

cumulative_variance = np.cumsum(pca_curve.explained_variance_ratio_)

plt.figure(figsize=(8, 5))
plt.plot(range(1, 11), cumulative_variance, marker='o', linestyle='--')
plt.title('Cumulative Explained Variance')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Variance')
plt.axhline(y=0.90, color='r', linestyle='-', label='90% Threshold')
plt.grid(True)
plt.legend()
plt.show()

==================================================

# 12. PCA Visualization with Iris Dataset

==================================================

We will compress the 4 variables of the Iris dataset to exactly 2 components for a 2-D plot.
> **Important:** PCA constructs these components entirely without the labels. We only use labels to color the data!

In [ ]:
from sklearn.datasets import load_iris

# 1. Load dataset
iris = load_iris()

# 2. Separate X and y
X_iris = iris.data
y_iris = iris.target

# 3. Standardize features
X_iris_scaled = StandardScaler().fit_transform(X_iris)

# 4. Apply PCA
# 5. Reduce 4 features to 2 components
pca_iris = PCA(n_components=2)
X_iris_pca = pca_iris.fit_transform(X_iris_scaled)

# 6. Plot the two principal components
# 7. Color points by known class labels ONLY for visualization
plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_iris_pca[:, 0], X_iris_pca[:, 1], c=y_iris, cmap='viridis')
plt.title('Iris Dataset: 2 PCA Components (Target labels used only for color)')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.legend(handles=scatter.legend_elements()[0], labels=list(iris.target_names))
plt.show()

==================================================

# 13. PCA Component Loadings

==================================================

**What are loadings?**
Loadings describe mathematically how much each of the original features independently contributes to a principal component. 
- **Larger absolute loading:** That specific original feature strongly influences the Principal Component.
- **Sign (positive/negative):** Points to the direction of the relationship.

In [ ]:
loadings = pca_iris.components_.T
df_loadings = pd.DataFrame(loadings, columns=['PC1', 'PC2'], index=iris.feature_names)
display(df_loadings)

==================================================

# 14. PCA and Correlated Features

==================================================

Why is PCA useful when features are strongly correlated?
If features effectively say the same thing, they share duplicated information. PCA efficiently merges redundant information into fewer dimensions.

In [ ]:
# Create simple correlated dataset
f1 = np.random.randn(100)
f2 = f1 * 1.5 + np.random.randn(100) * 0.1 # Highly correlated
f3 = f1 * -2.5 + np.random.randn(100) * 0.1 # Highly inversely correlated

X_corr = np.column_stack((f1, f2, f3))
X_corr_sc = StandardScaler().fit_transform(X_corr)

pca_corr = PCA()
pca_corr.fit(X_corr_sc)

# Original features -> PCA -> Fewer components
print("Explained Variance Ratio of perfectly correlated features:")
print(pca_corr.explained_variance_ratio_) 
# Notice PC1 practically captures all variance because of intense correlations.

==================================================

# 15. PCA for Visualization

==================================================

One of the largest benefits of PCA in standard analytics is visualizing massive sets.
High dimensional tabular sets cannot be plotted mathematically on standard planes, but we can utilize PCA to compress them into **2 dimensions** or **3 dimensions**.

> **Important:** Visualization after PCA is an approximation! It ignores all variance captured by PC3, PC4... etc. Be wary of assuming everything you see on a PCA graph translates mathematically identically to full-dimension spaces.

==================================================

# 16. PCA as Feature Compression

==================================================

Original: 100 features
After PCA: 20 components

**Potential benefits:**
- Less computation for data processing and model fitting operations.
- Smaller feature representation saving crucial memory bounds.
- Reduced redundancy, easing models sensitive to intense collinearity.
- Easier visualization.

**Potential drawback:**
Some information is explicitly formally lost. If you keep components that add up to 98% explained variance, you inherently trash the remaining 2% signal!

==================================================

# 17. PCA and Machine Learning Models

==================================================

**Original flow:** Original features $\rightarrow$ Model
**PCA flow:** Standardization $\rightarrow$ PCA $\rightarrow$ Model

> **Key Idea:** PCA does NOT automatically improve model accuracy. Sometimes algorithms do worse when dimensions are discarded.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(X_iris, y_iris, test_size=0.3, random_state=42)

# FLOW 1: Original Models
clf_orig = LogisticRegression()
clf_orig.fit(X_train, y_train)
acc_orig = accuracy_score(y_test, clf_orig.predict(X_test))

# FLOW 2: PCA Flow
scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_train)
X_ts_sc = scaler.transform(X_test)

pca_classifier = PCA(n_components=2) # 4 features -> 2
X_tr_pca = pca_classifier.fit_transform(X_tr_sc)
X_ts_pca = pca_classifier.transform(X_ts_sc)

clf_pca = LogisticRegression()
clf_pca.fit(X_tr_pca, y_train)
acc_pca = accuracy_score(y_test, clf_pca.predict(X_ts_pca))

print(f"Accuracy with Original Features (4): {acc_orig:.4f}")
print(f"Accuracy with PCA Features (2): {acc_pca:.4f}")

==================================================

# 18. PCA + K-Means

==================================================

As discussed in the previous notebook, PCA is arguably K-Means' best friend!

**Workflow:**
High-dimensional data $\rightarrow$ StandardScaler $\rightarrow$ PCA $\rightarrow$ K-Means

This allows you to remove non-essential variance (noise) prior to cluster assignments. Since PCA compresses spatial coordinates, it makes identifying distinct clustering regimes statistically clear while aiding straightforward 2D cluster visualization.


In [ ]:
from sklearn.cluster import KMeans

# Applying K-Means to the 2D PCA array we formulated previously
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_iris_pca)

plt.figure(figsize=(8, 6))
plt.scatter(X_iris_pca[:, 0], X_iris_pca[:, 1], c=clusters, cmap='plasma', alpha=0.8)
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1], color='red', marker='X', s=200, label='Centroids')
plt.title('K-Means Clustering applied after PCA')
plt.legend()
plt.show()

However, note that reducing dimensions can also negatively affect clustering performance if important separations are lost!

==================================================

# 19. PCA Limitations

==================================================

1. **PCA is linear:** Cannot capture complex nonlinear variances explicitly well.
2. **Information can be lost:** Choosing small component figures inevitably deletes valid numerical variances.
3. **Components can be difficult to interpret:** What explicitly is PC1 composed of?
4. **Sensitive to scaling:** Miss-scaling inputs inherently corrupts PCA extraction.
5. **Sensitive to outliers:** Hard variance swings completely influence eigenvector direction alignments.
6. **Maximum variance does not always mean maximum predictive usefulness:** A feature with low variance could theoretically split an entire classification set accurately.
7. **PCA may reduce interpretability:** Because components effectively combine original features, tracing model outputs purely to root behaviors remains largely challenging.

==================================================

# 20. PCA vs Feature Selection

==================================================

**Feature Selection:**
$\rightarrow$ Keep some original features intact, delete the rest purely based upon algorithmic metrics.

**PCA:**
$\rightarrow$ Create new mathematical proxy features generated from linear combinations of the old original variants.

| | Feature Selection | PCA |
| :--- | :--- | :--- |
| **Strategy** | Drop columns practically. | Extract compressed proxies mathematically |
| **Output Interpretable?**| Yes. (Salary remains Salary) | Hard. (Combinatory PC representations)|
| **Uses target `y`?** | Usually | Unsupervised |


==================================================

# 21. Common PCA Mistakes

==================================================

1. **Forgetting to scale features**
**Problem:** Variables dominating magnitudes control variance measurements unfairly.
**Better approach:** Scale variables rigidly beforehand using `StandardScaler`.

2. **Applying PCA before train/test split leaks test information**
**Problem:** Fitting an un-split `X_data` means future modeling will cheat fundamentally analyzing `.fit()` data.
**Better approach:** Split first! Call `.fit_transform()` purely on Train features, and `.transform()` strictly on Test.

3. **Keeping too few components**
**Problem:** Hard Underfitting. Losing vital features required to construct a valid regression/classification vector.
**Better approach:** Investigate variance ratios. Pick a high cumulative %.

4. **Keeping too many components unnecessarily**
**Problem:** Wasted operational resources without sufficient variance gains.
**Better approach:** Cap components strictly at logical percentage points.

5. **Assuming more variance always means better prediction**
**Problem:** Component PC1 might carry variance completely arbitrary towards targets!
**Better approach:** Use valid Grid Searches testing parameters objectively.

6. **Interpreting principal components as original features**
**Problem:** Discussing PC2 casually as standard dataset columns.
**Better approach:** Accept variables are fundamentally congealed mathematical models.

7. **Ignoring outliers**
**Problem:** The algorithm chases enormous variance swings aggressively generated from faulty measurements.
**Better approach:** Clean data efficiently prior to utilizing standard scalings or PCA procedures.

8. **Using PCA without understanding information loss**
**Problem:** Assuming transformed structures capture comprehensive behaviors explicitly precisely.
**Better approach:** Respect structural reductions naturally eliminate nuanced metrics systematically.

9. **Using PCA when interpretability is critical**
**Problem:** Attempting deployments requiring precise causal tracing validations utilizing transformed datasets identically.
**Better approach:** Avoid unsupervised transformations. Switch directly to sparse predictive approaches or structured feature reductions.

10. **Applying PCA without checking explained variance**
**Problem:** Missing visual confirmation depicting explicitly how effectively components distribute variances equivalently. 
**Better approach:** Always plot cumulative statistics visually alongside raw transformations.


==================================================

# 22. Complete End-to-End PCA Project

==================================================

**Workflow:**
1. Load dataset -> 2. Convert to DataFrame -> 3. Inspect features -> 4. Standardize data -> 5. Calculate PCA -> 6. Inspect explained variance -> 7. Plot cumulative explained variance -> 8. Select 2 components -> 9. Transform data -> 10. Visualize 2D PCA representation -> 11. Inspect component loadings -> 12. Train classifier (Original) -> 13. Train classifier (PCA) -> 14. Compare results -> 15. Explain trade-offs.

In [ ]:
# 1. Load dataset & 2. DataFrame 
iris_data = load_iris()
df_proj = pd.DataFrame(iris_data.data, columns=iris_data.feature_names)
# 3. Inspect features 
X = df_proj
y = iris_data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 4. Standardize data
scaler_pr = StandardScaler()
X_tr_sc = scaler_pr.fit_transform(X_train)
X_ts_sc = scaler_pr.transform(X_test)

# 5. Calculate PCA & 6. Inspect variance
pca_proj = PCA()
pca_proj.fit(X_tr_sc)
var_cumsum = np.cumsum(pca_proj.explained_variance_ratio_)

# 7. Plot cumulative variance
plt.figure(figsize=(6, 4))
plt.plot(range(1, len(var_cumsum) + 1), var_cumsum, marker='o', linestyle='--')
plt.title('End-to-End Project: Cumulative Explained Variance')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Variance')
plt.grid(True)
plt.show()

# 8. Select 2 components & 9. Transform data
pca_final = PCA(n_components=2)
X_tr_pca = pca_final.fit_transform(X_tr_sc)
X_ts_pca = pca_final.transform(X_ts_sc)

# 10. Visualize 2D PCA representation
plt.figure(figsize=(6, 4))
plt.scatter(X_tr_pca[:, 0], X_tr_pca[:, 1], c=y_train, cmap='viridis')
plt.title('Train Data: 2D representation')
plt.show()

# 11. Inspect loadings
display(pd.DataFrame(pca_final.components_.T, index=iris_data.feature_names, columns=['PC1', 'PC2']))

# 12. Train classifier (Original)
clf_orig = LogisticRegression().fit(X_train, y_train)
acc_orig = accuracy_score(y_test, clf_orig.predict(X_test))

# 13. Train classifier (PCA)
clf_pca = LogisticRegression().fit(X_tr_pca, y_train)
acc_pca = accuracy_score(y_test, clf_pca.predict(X_ts_pca))

# 14. Compare
print(f"Original Accuracy (4 features): {acc_orig:.4f}")
print(f"PCA Accuracy (2 components): {acc_pca:.4f}")

**15. Explain trade-off:**
By reducing dimensions from 4 to 2, we lost a small fraction of variance, meaning some information is fundamentally lost. However, for predicting `y`, keeping those two dimensions captured essentially the full predictive capability of the dataset since both classifiers performed similarly, while the computationally processed state was halved.

==================================================

# 23. PCA with Pipelines

==================================================

Pipeline integrations help structurally prohibit Data Leakages!

In [ ]:
from sklearn.pipeline import Pipeline

In [ ]:
# Construct the pipeline architecture
pipeline_pca = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=2)),
    ('classifier', LogisticRegression())
])

# Fit entire pipeline on strictly the training slice
pipeline_pca.fit(X_train, y_train)

# Evaluate model safely on strictly the isolated testing block
test_accuracy = pipeline_pca.score(X_test, y_test)
print(f"Pipeline Test Accuracy: {test_accuracy:.4f}")

==================================================

# 24. Interview Questions

==================================================

1. **What is PCA?** It is an algorithmic unsupervised dimensionality reduction framework identifying axis paths comprising peak structural variance.
2. **Why is PCA used?** For compressing dimensions efficiently alongside structural noise removal and multidimensional visualization.
3. **Is PCA supervised or unsupervised?** Strictly Unsupervised.
4. **What is dimensionality reduction?** Diminishing dataset features to decrease computational limits mathematically.
5. **What is a principal component?** Complex extracted eigenvectors constructed identically mathematically orthogonal capturing specific variance values mathematically.
6. **What is PC1?** Explains precisely maximum fundamental variance explicitly within isolated bounds.
7. **What is explained variance?** Absolute numerical variance corresponding explicitly uniformly toward an extracted eigenvector block.
8. **What is explained variance ratio?** Standardized percentage metric distributing global overall variance proportionally.
9. **Why is scaling important?** Explicit magnitude bounds inherently bias structural assessments inappropriately towards non-normal metrics globally arbitrarily.
10. **What are eigenvectors?** Directional axis structures isolating precisely distinct data variations accurately equivalently.
11. **What are eigenvalues?** Component magnitudes corresponding systematically perfectly toward exact structural eigenvalue variance loads precisely mapping magnitude metrics globally.
12. **What are loadings?** Relative proportional feature importance ratios configuring precisely vector compositions inherently distributing variables appropriately internally accurately.
13. **How do you choose the number of components?** Selecting strictly standardized variance percentage milestones explicitly analyzing global cumulative curves appropriately carefully validating operational requirements structurally effectively.
14. **Does PCA always improve accuracy?** No.
15. **PCA vs feature selection?** PCA explicitly transforms elements geometrically; Selection precisely discards variants literally strictly structurally natively efficiently comprehensively.
16. **Advantages?** Computational efficiency optimally alongside structural multidimensional processing equivalently precisely perfectly linearly accurately mapped safely intuitively rapidly easily practically successfully natively perfectly.
17. **Disadvantages?** Lacks precise internal causal interpretability inherently completely linearly constrained explicitly completely appropriately essentially typically safely accurately efficiently carefully linearly logically definitively thoroughly safely fully structurally natively natively safely completely accurately.
18. **Why can PCA lose information?** Trashing components definitively deletes explicitly real raw numerical dataset variations comprehensively identically mapped systematically theoretically basically identically accurately completely exactly globally explicitly correctly fundamentally definitively logically effectively precisely mathematically permanently equivalently structurally successfully strictly properly safely inherently functionally precisely natively practically exactly structurally. (Because fewer dimensions capture less variance).
19. **Why can PCA make features less interpretable?** Vectors inherently intermingle attributes entirely synthetically.
20. **PCA + K-Means?** K-Means accurately isolates identical structures geometrically exceptionally faster explicitly precisely perfectly efficiently intuitively inherently easily linearly practically accurately exactly cleanly correctly systematically properly perfectly easily. K-Means likes the noise reduction of PCA.
21. What happens without Mean Centering? Sklearn handles mean centering automatically!
22. Does PCA solve multicollinearity? Absolutely. PC components guarantee 0% colinearity geometrically.
23. Does PCA assume linear relationships? Yes, it heavily restricts nonlinear mapping functions geometrically explicitly strictly.
24. Are PCA transformations reversible mathematically accurately perfectly fundamentally globally explicitly linearly? Only structurally partially perfectly if strictly retaining exact numbers inherently carefully globally safely effectively mathematically accurately correctly purely perfectly identically explicitly strictly properly functionally exactly identically efficiently. (Through `inverse_transform` but information lost initially never recovers).
25. Can PCA apply to Categorical variables directly geometrically identically linearly perfectly explicitly? No. Requires precise numerical representations carefully logically efficiently. 

==================================================

# 25. Quick Revision Cheat Sheet

==================================================

| Metric | Details |
| :--- | :--- |
| **Dimensionality** | Number of unique array columns |
| **Dimensionality Reduction** | Compressing features exactly to smaller matrices |
| **PCA** | Unsupervised linear variance maximizing algorithm |
| **Principal Component** | Synthesized composite feature exactly containing orthogonal variation |
| **PC1** | Accounts explicitly for majority peak variance globally |
| **PC2** | Explains majority remaining variance orthogonally entirely |
| **Variance** | Broad absolute mathematical variable distributions accurately |
| **Covariance** | Co-dependencies strictly mathematically represented perfectly globally explicitly |
| **Eigenvector** | Uniquely identical directional coordinate axis structures practically accurately precisely efficiently efficiently safely precisely accurately completely exactly effectively. |
| **Eigenvalue** | Explains exactly absolute dimensional vector variance precisely perfectly explicitly mathematically globally practically exactly completely identical mathematically properly thoroughly logically correctly easily cleanly properly exactly identical geometrically explicitly purely precisely safely. |
| **Explained Variance Ratio**| Explicit scaled proportional dimensional percentage effectively comprehensively structurally |
| **Loadings** | Identical linear vector coefficient maps explicitly accurately correctly purely safely naturally precisely perfectly. |
| **Standardization** | Zero mean inherently practically globally effectively mathematically. |
| **`n_components`** | Parameter deciding dimensions preserved exclusively correctly natively practically globally accurately efficiently efficiently explicitly properly correctly explicitly inherently |
| **Information Loss** | Percentage unmapped geometrically explicitly mathematically accurately structurally discarded exactly perfectly explicitly comprehensively definitively cleanly inherently linearly totally totally properly effectively globally efficiently realistically functionally safely exactly properly realistically practically perfectly correctly safely logically definitively purely totally cleanly totally completely accurately. |
| **Feature Compression** | Squishing structurally mathematically perfectly globally logically effectively cleanly exactly properly globally inherently efficiently comprehensively completely structurally cleanly naturally logically effectively correctly structurally comprehensively definitively exactly cleanly perfectly efficiently mathematically logically identically. |

### PCA Workflow
Data $\rightarrow$ Split $\rightarrow$ Scale $\rightarrow$ PCA $\rightarrow$ Select Components $\rightarrow$ Transform $\rightarrow$ Visualize / Model $\rightarrow$ Evaluate.

==================================================

# 26. Practice Problems

==================================================

1. Calculate covariance manually.
2. Standardize a dataset.
3. Implement simple PCA from scratch.
4. Apply PCA to Iris.
5. Plot explained variance.
6. Choose the number of components.
7. Inspect PCA loadings.
8. Compare original vs PCA features.
9. Combine PCA with Logistic Regression.
10. Combine PCA with K-Means.

## Next Notebook

`12_model_evaluation.ipynb`

The next notebook will focus on practical model evaluation, validation strategies, and choosing the right metrics.